# 固定42問題・4モデル比較
Qwen2.5 1.5B/3B/7BとPhi-3.5-mini。全FP16、共通system・生成設定。問題と採点は変更しない。PLAN.md参照。

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install transformers==4.57.6 accelerate scikit-learn scipy pandas numpy sentencepiece


In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/GraduationThesis-model-comparison')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ryouy/GraduationThesis.git',str(REPO)],check=True)
OUT=Path('/content/model-comparison-20261104')
OUT.mkdir(exist_ok=True)
subprocess.run([sys.executable,'1104_model_comparison/run.py','--resolve','--out',str(OUT)],cwd=REPO,check=True)
models=json.loads((OUT/'resolved_models.json').read_text())
for entry in models:
    key=entry['key']
    proc=subprocess.Popen([sys.executable,'-u','1104_model_comparison/run.py','--key',key,'--out',str(OUT)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with (OUT/(key+'.log')).open('w') as log:
        for line in proc.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
    print('PROCESS EXIT',key,proc.wait(),flush=True)
print('ALL MODELS ATTEMPTED',flush=True)


In [ ]:
print('COMPARISON_EXPORT_START')
print(json.dumps({str(p.relative_to(OUT)):p.read_text() for p in sorted(OUT.rglob('*')) if p.is_file()}))
print('COMPARISON_EXPORT_END')